### 05 — Multimodal Fusion using Bidirectional Cross-Attention

What this notebook does:

- load the existing dataset splits
- rebuild the clinical-text representation used by ClinicalBERT
- load the trained U-Net, EfficientNet image encoder, and ClinicalBERT encoder
- freeze the pretrained feature extractors
- generate predicted lesion masks for a small batch of ultrasound images
- extract lesion-guided visual tokens using EfficientNet-B0
- extract clinical token features using ClinicalBERT
- confirm that the image and clinical batches are aligned
- combine both modalities using bidirectional cross-attention
- pool the fused visual and clinical token features
- concatenate both pooled representations into one multimodal feature vector.

The final output is a 1536-dimensional multimodal representation that will be used by the classification and report-generation components.


In [ ]:
from pathlib import Path
import sys

import pandas as pd
import numpy as np

from PIL import Image

import torch
import torch.nn as nn

from transformers import AutoTokenizer

PROJECT_ROOT = Path("..").resolve()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.unet import UNet
from src.image_encoder import EfficientNetEncoder
from src.clinical_encoder import ClinicalBERTEncoder
from src.fusion import CrossAttentionFusion


### 1. Define Project Paths

The notebook uses the existing project folders for the raw ultrasound images, processed dataset splits, and trained model checkpoints.

The notebook also selects CUDA when a compatible GPU is available.


In [ ]:
DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"

IMAGE_DIR = RAW_DIR / "images"
MODEL_DIR = PROJECT_ROOT / "models"

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

### 2. Load the Dataset Splits


In [ ]:
train_df = pd.read_csv(
    PROCESSED_DIR / "train.csv"
)

val_df = pd.read_csv(
    PROCESSED_DIR / "val.csv"
)

test_df = pd.read_csv(
    PROCESSED_DIR / "test.csv"
)

print("Train:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))

### 3. Create the Clinical Text

The structured clinical variables are converted into the short text format used in the ClinicalBERT notebook


In [ ]:
def create_clinical_text(row):
    return (
        f"Patient age: {row['Age']} years. "
        f"Tissue composition: {row['Tissue_composition']}. "
        f"Clinical signs: {row['Signs']}. "
        f"Symptoms: {row['Symptoms']}."
    )

In [ ]:
train_df["clinical_text"] = train_df.apply(
    create_clinical_text,
    axis=1
)

In [ ]:
print(train_df.iloc[0]["clinical_text"])

### 4. Load the ClinicalBERT Tokenizer

The tokenizer for `emilyalsentzer/Bio_ClinicalBERT` is loaded so that the clinical text can be converted into token IDs and attention masks.


In [ ]:
MODEL_NAME = "emilyalsentzer/Bio_ClinicalBERT"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

### 5. Load ClinicalBERT

The reusable `ClinicalBERTEncoder` is initialised and moved to the selected device.

In [ ]:
clinical_encoder = ClinicalBERTEncoder(
    model_name=MODEL_NAME
).to(device)

### 6. Freeze ClinicalBERT

The pretrained ClinicalBERT parameters are frozen so that the encoder is used only for feature extraction during this stage.

In [ ]:
for parameter in clinical_encoder.parameters():
    parameter.requires_grad = False

clinical_encoder.eval()

### 7. Load the Trained U-Net

The trained U-Net segmentation model is loaded from the saved checkpoint.


In [ ]:
segmentation_model = UNet().to(device)

In [ ]:
segmentation_model.load_state_dict(
    torch.load(
        MODEL_DIR / "unet_baseline.pth",
        map_location=device
    )
)

### 8. Freeze the U-Net

The U-Net parameters are frozen because the segmentation model has already been trained, it is used only to predict lesion masks.


In [ ]:
for parameter in segmentation_model.parameters():
    parameter.requires_grad = False

segmentation_model.eval()

print("U-Net loaded successfully")

### 9. Load the EfficientNet Image Encoder

The reusable EfficientNet-B0 image encoder is loaded with an output size of `768`.


In [ ]:
image_encoder = EfficientNetEncoder(
    output_dim=768
).to(device)

### 10. Freeze the EfficientNet Backbone

The pretrained EfficientNet convolutional feature extractor is frozen.

In [ ]:
for parameter in image_encoder.features.parameters():
    parameter.requires_grad = False

The final projection layer remains trainable so that the visual features can adapt to the multimodal task.


In [ ]:
for parameter in image_encoder.projection.parameters():
    parameter.requires_grad = True

### 11. Load a Small Batch of Ultrasound Images

Test the complete multimodal pipeline before using it in the classification stage.


In [ ]:
BATCH_SIZE = 4

sample_df = train_df.iloc[:BATCH_SIZE].copy()

A helper function loads each ultrasound image, converts it to grayscale, resizes it to `224 × 224`, normalises it to the range `0–1`, and adds the channel dimension expected by U-Net.


In [ ]:
def load_ultrasound_image(
    filename,
    image_dir,
    image_size=224
):
    image_path = image_dir / filename

    image = Image.open(
        image_path
    ).convert("L")

    image = image.resize(
        (image_size, image_size),
        Image.BILINEAR
    )

    image = np.array(
        image,
        dtype=np.float32
    ) / 255.0

    image = torch.tensor(
        image,
        dtype=torch.float32
    ).unsqueeze(0)

    return image

The selected ultrasound images are loaded and stacked into one batch tensor.


In [ ]:
images = []

for filename in sample_df["Image_filename"]:
    image = load_ultrasound_image(
        filename,
        IMAGE_DIR
    )

    images.append(image)

In [ ]:
images = torch.stack(images)
print(images.shape)

In [ ]:
images = images.to(device)

### 12. Generate Predicted Lesion Masks

The ultrasound batch is passed through the U-Net.

The predicted masks are created from the segmentation probabilities and are later used to guide the EfficientNet visual features toward the lesion region.


In [ ]:
with torch.no_grad():
    segmentation_logits = segmentation_model(
        images
    )

    predicted_masks = torch.sigmoid(
        segmentation_logits
    )

    predicted_masks = (
        predicted_masks > 0.5
    ).float()

In [ ]:
print(predicted_masks.shape)

### 13. Prepare the Images for EfficientNet-B0

EfficientNet expects three-channel images, so each grayscale ultrasound image is repeated across three channels.

The images are then normalised using the ImageNet mean and standard deviation values used by the pretrained EfficientNet-B0 model.


In [ ]:
efficientnet_images = images.repeat(
    1, 3, 1, 1
)

In [ ]:
mean = torch.tensor(
    [0.485, 0.456, 0.406],
    device=device
).view(1, 3, 1, 1)

std = torch.tensor(
    [0.229, 0.224, 0.225],
    device=device
).view(1, 3, 1, 1)

In [ ]:
efficientnet_images = (
    efficientnet_images - mean
) / std

### 14. Generate the Visual Tokens

The ultrasound images and predicted lesion masks are passed through the EfficientNet image encoder.


In [ ]:
visual_tokens = image_encoder(
    efficientnet_images,
    predicted_masks
)

In [ ]:
print("Visual tokens:", visual_tokens.shape)

### 15. Tokenise the Clinical Text

The clinical-text are tokenised using ClinicalBERT. 

Padding is applied so that all sequences in the batch have the same length, and the maximum sequence length is limited to 64 tokens.


In [ ]:
texts = sample_df[
    "clinical_text"
].tolist()

In [ ]:
for text in texts:
    print(text)
    print()

In [ ]:
clinical_batch = tokenizer(
    texts,
    padding=True,
    truncation=True,
    max_length=64,
    return_tensors="pt"
)

In [ ]:
clinical_batch = {
    key: value.to(device)
    for key, value in clinical_batch.items()
}

### 16. Generate ClinicalBERT Tokens

The tokenised clinical batch is passed through the ClinicalBERT encoder.

The output contains contextual clinical features with 768 dimensions per token.


In [ ]:
with torch.no_grad():
    clinical_tokens = clinical_encoder(
        input_ids=clinical_batch["input_ids"],
        attention_mask=clinical_batch[
            "attention_mask"
        ],
        token_type_ids=clinical_batch.get(
            "token_type_ids"
        )
    )

In [ ]:
print(
    "Clinical tokens:",
    clinical_tokens.shape
)

### 17. Confirm Batch Alignment

Before fusion, check that the visual-token batch and clinical-token batch contain the same number of cases.



In [ ]:
print(
    "Visual batch size:",
    visual_tokens.shape[0]
)

print(
    "Clinical batch size:",
    clinical_tokens.shape[0]
)

### 18. Create the Cross-Attention Fusion Model

The CrossAttentionFusion module combines the visual and clinical token sequences.

It uses bidirectional multi-head attention:

- visual tokens attend to the clinical tokens
- clinical tokens attend to the visual tokens.

Both modalities use an embedding size of 768, with eight attention heads.


In [ ]:
fusion_model = CrossAttentionFusion(
    embed_dim=768,
    num_heads=8,
    dropout=0.1
).to(device)

### 19. Create the Clinical Padding Mask

Clinical sequences have different lengths, so padded token positions must be ignored during attention.


In [ ]:
clinical_padding_mask = (
    clinical_batch["attention_mask"] == 0
)

In [ ]:
print(
    clinical_padding_mask.shape
)

### 20. Run Bidirectional Cross-Attention

The visual and clinical tokens are passed through the fusion model.
The output consists of:

- fused visual tokens containing clinical context
- fused clinical tokens containing visual context.


In [ ]:
visual_fused, clinical_fused = (
    fusion_model(
        visual_tokens,
        clinical_tokens,
        clinical_padding_mask=
            clinical_padding_mask
    )
)

In [ ]:
print(
    "Fused visual:",
    visual_fused.shape
)

print(
    "Fused clinical:",
    clinical_fused.shape
)

### 21. Pool the Fused Visual Features

The fused visual tokens are averaged across the spatial-token dimension to produce one 768-dimensional visual vector for each case.


In [ ]:
visual_vector = visual_fused.mean(
    dim=1
)

In [ ]:
print(visual_vector.shape)

### 22. Pool the Fused Clinical Features

Clinical sequences contain padding, so a masked mean is used instead of a normal average. 

So that only valid ClinicalBERT tokens contribute to the clinical vector.


In [ ]:
def masked_mean_pool(
    tokens,
    attention_mask
):
    mask = (
        attention_mask
        .unsqueeze(-1)
        .float()
    )

    summed = (
        tokens * mask
    ).sum(dim=1)

    counts = mask.sum(
        dim=1
    ).clamp(min=1e-9)

    return summed / counts

In [ ]:
clinical_vector = masked_mean_pool(
    clinical_fused,
    clinical_batch["attention_mask"]
)

In [ ]:
print(clinical_vector.shape)

### 23. Combine the Visual and Clinical Features

The pooled visual and clinical vectors are concatenated:

```text
Visual vector:   768 dimensions
Clinical vector: 768 dimensions
        ↓
Multimodal vector: 1536 dimensions
```


In [ ]:
multimodal_features = torch.cat(
    [
        visual_vector,
        clinical_vector
    ],
    dim=1
)

In [ ]:
print(
    "Multimodal features:",
    multimodal_features.shape
)

### 24. Check the Final Tensor Shapes

Make sure the complete multimodal pipeline is producing the expected dimensions.


In [ ]:
print(
    "Images:",
    images.shape
)

print(
    "Predicted masks:",
    predicted_masks.shape
)

print(
    "Visual tokens:",
    visual_tokens.shape
)

print(
    "Clinical tokens:",
    clinical_tokens.shape
)

print(
    "Fused visual:",
    visual_fused.shape
)

print(
    "Fused clinical:",
    clinical_fused.shape
)

print(
    "Multimodal features:",
    multimodal_features.shape
)